# HAM10000: Hudcancerklassificering
Dataset: [Skin Cancer MNIST: HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000) (Kaggle)

Detta är ett akademiskt övningsprojekt, inte ett kliniskt diagnosverktyg. HAM10000 är dessutom insamlat främst från österrikiska och australiensiska populationer, vilket innebär en känd skevhet mot ljusare hudtoner. Riktiga dermatologi-AI-verktyg kräver omfattande klinisk validering innan de kan användas skarpt.

## 1. Ladda data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    precision_recall_curve, auc
)
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import binom

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

`tf.random.set_seed(42)` fångar den största källan till variation mellan körningar, de nya lagrens slumpmässiga startvikter. Garanterar inte bitidentiska resultat (GPU-beräkningar har egen icke-determinism även med ett fixerat frö), men minskar variationen kraftigt jämfört med att inte sätta något frö alls.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Path to dataset files:", path)

In [ ]:
print(os.listdir(path))

Sju filer utöver metadatan och bildmapparna är förnedskalade `hmnist_*.csv`-filer (8x8 och 28x28 pixlar, utplattade). De används inte här. Hudlesion-diagnostik bygger på kant- och texturdetaljer som är utraderade vid den upplösningen, riktiga JPG-bilder ger ett mycket bättre underlag.

In [ ]:
df = pd.read_csv(os.path.join(path, "HAM10000_metadata.csv"))
print(df.shape)
df.head()

### Bildmappar
Datasetet innehåller både `HAM10000_images_part_1` och `ham10000_images_part_1` (skillnad bara i versaler). Verifierar att det är samma bilder dubblerade under två mappnamn innan de kopplas till metadatan.

In [ ]:
files_upper = set(os.listdir(os.path.join(path, "HAM10000_images_part_1")))
files_lower = set(os.listdir(os.path.join(path, "ham10000_images_part_1")))
print("Antal filer:", len(files_upper), len(files_lower))
print("Är innehållet identiskt?", files_upper == files_lower)

In [ ]:
image_dir_1 = os.path.join(path, "HAM10000_images_part_1")
image_dir_2 = os.path.join(path, "HAM10000_images_part_2")

image_paths = {}
for d in [image_dir_1, image_dir_2]:
    for fname in os.listdir(d):
        image_id = fname.replace('.jpg', '')
        image_paths[image_id] = os.path.join(d, fname)

df['image_path'] = df['image_id'].map(image_paths)
print("Saknade bildsökvägar:", df['image_path'].isnull().sum())

In [ ]:
df.info()

## 2. Rengör data

In [ ]:
print(df['sex'].value_counts())
print(df['localization'].value_counts())

`sex` och `localization` har en egen `unknown`-kategori, inga `NaN`, kräver ingen imputering. `age` har 57 saknade värden och behöver hanteras separat.

In [ ]:
missing_age_rows = df[df['age'].isnull()]
print("Sex-fördelning bland raderna med saknad ålder:")
print(missing_age_rows['sex'].value_counts())

47 av de 57 raderna med saknad ålder har också okänt kön, samma undergrupp, sannolikt en batch anonymiserade patientjournaler. 10 rader (6 man, 4 kvinna) saknar bara ålder, ordinärt bortfall. Medianimputering är väl motiverat, 57 rader är 0,57% av datan.

In [ ]:
df['age'] = df['age'].fillna(df['age'].median())
print(df['age'].isnull().sum())

In [ ]:
malignant = ['mel', 'bcc', 'akiec']
df['malignant'] = df['dx'].isin(malignant).astype(int)
print(df['malignant'].value_counts())
print(df['malignant'].value_counts(normalize=True).round(3) * 100)

Binär målvariabel: `mel` (melanom), `bcc` (basalcellscancer) och `akiec` (aktinisk keratos, förstadium till cancer) räknas som malign, resten benign. 80,5% benign mot 19,5% malign, obalanserat men hanterbart, betydligt bättre än 7-klassversionens minsta klass på bara 1,1%.

## 3. EDA: statistiska summeringar och relationer

In [ ]:
print(df['dx'].value_counts())
print(df['dx'].value_counts(normalize=True).round(3) * 100)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

df['dx'].value_counts().plot(kind='bar', ax=axes[0], color='#2a78d6')
axes[0].set_title('Fördelning per diagnos (7 klasser)')
axes[0].set_ylabel('Antal bilder')

df['malignant'].map({0:'Benign', 1:'Malignant/pre-malignant'}).value_counts().plot(kind='bar', ax=axes[1], color=['#2a78d6','#e34948'])
axes[1].set_title('Fördelning, binär')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

`nv` (melanocytiska nevi) dominerar med 66,9%, den kraftiga obalansen i 7-klassversionen är anledningen till att binär klassificering valdes som huvudspår.

In [ ]:
print(df.groupby('malignant')['age'].mean())
print(pd.crosstab(df['sex'], df['malignant'], normalize='index').round(3) * 100)

14 års åldersskillnad (49,1 mot 63,3 år), stämmer med känd epidemiologi, cancerrisk ökar med kumulativ solskada över tid. Män visar högre malign andel än kvinnor (22,7% mot 16,0%), också konsistent med känd statistik.

### Är noll maligna fall bland `sex = unknown` ett äkta mönster, eller bara ett litet stickprov?

In [ ]:
n_unknown = (df['sex']=='unknown').sum()
p_base = df['malignant'].mean()
p_zero = binom.pmf(0, n_unknown, p_base)
print(f"Antal med okänt kön: {n_unknown}")
print(f"Förväntat antal maligna vid basraten ({p_base:.3f}): {n_unknown * p_base:.1f}")
print(f"Sannolikhet att få exakt 0 av ren slump: {p_zero:.8f}")

Sannolikheten att få exakt noll maligna av 57 rent slumpmässigt är under en promille, ett äkta mönster, inte brus. Troligen en specifik datakälla eller studiebatch som bara bidrog med bekräftat benigna, anonymiserade fall, inte en klinisk slutsats om att okänt kön skulle skydda mot cancer.

In [ ]:
loc_malignant = df.groupby('localization')['malignant'].mean().sort_values(ascending=False) * 100
print(loc_malignant.round(1))
print()
print("Antal per lokalisation:")
print(df['localization'].value_counts())

Ansikte (42,7%), hjässa (36,7%) och öra (35,7%) toppar, kroppsdelar med mest kronisk solexponering. Bål (4,2%) lägst, minst solexponerat. `acral` (n=7) och `genital` (n=48) visar 0% men bygger på för lite data för att dra slutsatser från, samma logik som testet ovan för `sex = unknown`.

### Exempelbilder per diagnos

In [ ]:
dx_classes = sorted(df['dx'].unique())

fig, axes = plt.subplots(1, len(dx_classes), figsize=(21,4))
for ax, dx_class in zip(axes, dx_classes):
    sample = df[df['dx'] == dx_class].iloc[0]
    img = Image.open(sample['image_path'])
    ax.imshow(img)
    ax.set_title(f"{dx_class}\n({'malign' if sample['malignant']==1 else 'benign'})")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(dx_classes), 5, figsize=(15, 3*len(dx_classes)))
for row, dx_class in enumerate(dx_classes):
    samples = df[df['dx'] == dx_class].sample(5, random_state=42)
    for col, (_, sample) in enumerate(samples.iterrows()):
        img = Image.open(sample['image_path'])
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].text(-0.15, 0.5, dx_class, fontsize=14, fontweight='bold',
                                 ha='right', va='center', transform=axes[row, col].transAxes)
plt.tight_layout()
plt.show()

Fem exempel per klass, inte bara ett, visar genuin spridning inom varje diagnos. `mel` och `nv` ligger ofta visuellt nära varandra, konsekvent med varför tidig melanomdiagnos är kliniskt svår. `malign` är dessutom ingen enhetlig visuell kategori, `mel`, `bcc` och `akiec` ser mycket olika ut trots samma etikett.

## 4. Förbered bilddata för modellering

In [ ]:
IMG_SIZE = 128

def load_and_preprocess(path):
    img = Image.open(path).convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE))
    return np.array(img, dtype=np.float32) / 255.0

Bilder normaliseras till 0 till 1, `float32` uttryckligen istället för NumPys default `float64`, halverar minnesåtgången utan att tappa relevant precision. 128x128 är en avvägning mellan att bevara detaljer (t.ex. df:s dimple sign, mel:s blå-vita-slöja) och rimlig tränings-/minnestid.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['malignant']
)
print(train_df.shape, test_df.shape)
print(train_df['malignant'].value_counts(normalize=True).round(3))
print(test_df['malignant'].value_counts(normalize=True).round(3))

Dataframen delas innan några bilder laddas, inte efter. Att ladda alla 10015 bilder till en array och sedan dela den skulle temporärt kräva minne för originalarrayen plus båda delarna samtidigt, onödig risk för en krascht runtime. `stratify` säkerställer att båda delarna behåller samma 80,5/19,5-balans.

In [ ]:
X_train = np.array([load_and_preprocess(p) for p in train_df['image_path']])
y_train = train_df['malignant'].values

X_test = np.array([load_and_preprocess(p) for p in test_df['image_path']])
y_test = test_df['malignant'].values

print(X_train.shape, X_test.shape)
print(X_train.dtype)
print(f"X_train minne: {X_train.nbytes / 1e9:.2f} GB, X_test minne: {X_test.nbytes / 1e9:.2f} GB")

## 5. Bygg och träna CNN (transfer learning)

In [ ]:
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Rescaling(255.0)(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu', name='embedding_layer')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

cnn_model = models.Model(inputs=inputs, outputs=outputs)
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.summary()

Två detaljer värda att förstå. `layers.Rescaling(255.0)` finns för att EfficientNet, till skillnad från många andra förtränade nätverk, förväntar sig rå pixeldata i intervallet 0 till 255, inte 0 till 1. Utan den raden normaliseras bilderna dubbelt (en gång i `load_and_preprocess`, en gång till av EfficientNets egen interna Rescaling-lager), vilket krymper alla pixelvärden mot noll och gör att nätverket inte längre kan skilja bilder åt.

Modellen byggs med Functional API, inte Sequential, specifikt för att `cnn_model.input`/`cnn_model.output` ska vara tillgängliga senare när embeddings extraheras för klustringssteget. Namnet `cnn_model` används konsekvent genom hela notebooken och byggs aldrig om i en senare cell, ett generellt `model`-namn som återanvänds i flera celler är en vanlig källa till att av misstag träna över en redan tränad modell.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history = cnn_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

## 6. Utvärdera CNN

In [ ]:
y_pred_proba = cnn_model.predict(X_test).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(7,6))
plt.plot(recall, precision, label=f'PR-AUC = {pr_auc:.3f}')
plt.xlabel('Recall (andel faktiska maligna fall som fångas)')
plt.ylabel('Precision')
plt.title('Precision-Recall-kurva, malign klass')
plt.legend()
plt.show()

print("PR-AUC:", pr_auc)

Vid standardtröskeln 0,5 missas 174 av 391 faktiska maligna fall, 44,5%. ROC-AUC (0,891) ser bra ut, men är delvis en effekt av hur väl modellen hanterar den lätta majoritetsklassen. PR-AUC (0,660) fokuserar enbart på den maligna klassen och ger en ärligare bild av hur bra modellen faktiskt är på det som spelar roll. Ett nollskicklighetsval (gissa på basraten) hade gett runt 0,195 i PR-AUC, så 0,660 är klart över slumpen, men långt ifrån perfekt.

## 7. Kan modellen förbättras? Tröskeloptimering

En falsk positiv här kostar ett onödigt läkarbesök. En falsk negativ kostar en missad cancerdiagnos. Den asymmetrin motiverar att medvetet prioritera recall över precision, inte Youden's J som balanserar dem lika.

In [ ]:
for target in [0.80, 0.85, 0.90, 0.95]:
    idx = np.where(recall >= target)[0][-1]
    print(f"Recall >= {target}: tröskel={thresholds[idx]:.3f}, faktisk recall={recall[idx]:.3f}, precision={precision[idx]:.3f}")

Precisionen faller jämnt, ungefär 3,5 till 4,5 procentenheter per extra 5 procentenheter recall, ingen tydlig brant på vägen. Given att det här är ett screeningverktyg, inte en diagnos, väger en missad cancer tyngre än ett extra läkarbesök. 90% recall väljs som slutgiltig tröskel.

In [ ]:
target_recall = 0.90
idx = np.where(recall >= target_recall)[0][-1]
best_threshold = thresholds[idx]

y_pred_optimal = (y_pred_proba >= best_threshold).astype(int)
print(f"Vald tröskel: {best_threshold:.3f}")
print(confusion_matrix(y_test, y_pred_optimal))
print(classification_report(y_test, y_pred_optimal))

## 8. Kan modellen förbättras? Alternativa träningsstrategier
Två ytterligare försök testades. Kod och resultat dokumenteras här i sin helhet, av rigör, trots att ingen av dem förbättrade resultatet.

In [ ]:
weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
class_weight = {0: weights[0], 1: weights[1]}
print(class_weight)

### Försök 1: fryst bas, med class_weight

In [ ]:
base_model.trainable = False

inputs_w = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x_w = layers.Rescaling(255.0)(inputs_w)
x_w = base_model(x_w)
x_w = layers.GlobalAveragePooling2D()(x_w)
x_w = layers.Dense(128, activation='relu')(x_w)
x_w = layers.Dropout(0.3)(x_w)
outputs_w = layers.Dense(1, activation='sigmoid')(x_w)

model_weighted = models.Model(inputs=inputs_w, outputs=outputs_w)
model_weighted.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop_w = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history_weighted = model_weighted.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop_w],
    verbose=1
)

In [ ]:
y_pred_proba_w = model_weighted.predict(X_test).flatten()
precision_w, recall_w, _ = precision_recall_curve(y_test, y_pred_proba_w)
pr_auc_w = auc(recall_w, precision_w)
print("PR-AUC (fryst + class_weight):", pr_auc_w)

### Försök 2: finjustering av de sista 30 lagren, med class_weight

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

inputs_ft = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x_ft = layers.Rescaling(255.0)(inputs_ft)
x_ft = base_model(x_ft)
x_ft = layers.GlobalAveragePooling2D()(x_ft)
x_ft = layers.Dense(128, activation='relu')(x_ft)
x_ft = layers.Dropout(0.3)(x_ft)
outputs_ft = layers.Dense(1, activation='sigmoid')(x_ft)

model_finetuned = models.Model(inputs=inputs_ft, outputs=outputs_ft)
model_finetuned.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy', metrics=['accuracy']
)

early_stop_ft = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history_finetuned = model_finetuned.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop_ft],
    verbose=1
)

base_model.trainable = False  # återställ till fryst för resten av notebooken

In [ ]:
y_pred_proba_ft = model_finetuned.predict(X_test).flatten()
precision_ft, recall_ft, _ = precision_recall_curve(y_test, y_pred_proba_ft)
pr_auc_ft = auc(recall_ft, precision_ft)
print("PR-AUC (finjusterad + class_weight):", pr_auc_ft)

print()
print("Jämförelse:")
print(f"Fas 1, fryst, ingen viktning: {pr_auc:.3f}")
print(f"Fryst + class_weight: {pr_auc_w:.3f}")
print(f"Finjusterad + class_weight: {pr_auc_ft:.3f}")

Båda försöken presterar sämre än den ursprungliga, ovägda fas 1-modellen (0,660 mot 0,620 respektive 0,557). `class_weight` gör modellen mer beslutsam på den maligna klassen under träningen, men förvränger samtidigt sannolikhetsrangordningen den bygger på, motsatt effekt av vad tröskeljustering gör i efterhand, som flyttar beslutsgränsen utan att röra rangordningen alls. Eftersom tröskeljustering redan visat sig kunna nå 90% recall utan att kosta något i PR-AUC, var class-weighting ett försök att lösa ett redan löst problem på ett sätt som visade sig skada modellen istället. `cnn_model` (fas 1) förblir den slutgiltiga, valda modellen.

## 9. Unsupervised learning: utforska den tränade modellens representation
Målvariabeln finns redan (`malignant`), så unsupervised learning används här för att förstå datan snarare än att skapa en etikett: extrahera modellens interna inlärda representation (det tränade Dense-lagret, inte råa ImageNet-features), och undersöka om klustring utan facit ändå hittar en struktur som liknar den verkliga diagnosen.

In [ ]:
feature_extractor = models.Model(inputs=cnn_model.input, outputs=cnn_model.get_layer('embedding_layer').output)
feature_extractor.summary()

In [ ]:
X_all = np.concatenate([X_train, X_test])
dx_all = pd.concat([train_df['dx'], test_df['dx']]).values
malignant_all = np.concatenate([y_train, y_test])

embeddings = feature_extractor.predict(X_all, batch_size=32, verbose=1)
print(embeddings.shape)

In [ ]:
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings_scaled)
print("Förklarad varians:", pca.explained_variance_ratio_)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings_scaled)

In [ ]:
print(pd.crosstab(cluster_labels, malignant_all))
print()
print(pd.crosstab(cluster_labels, dx_all))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,6))

sns.scatterplot(x=coords[:,0], y=coords[:,1], hue=cluster_labels, palette='Set1', s=15, alpha=0.6, ax=axes[0])
axes[0].set_title('Klustring (unsupervised, k=2)')

sns.scatterplot(x=coords[:,0], y=coords[:,1], hue=malignant_all, palette='coolwarm', s=15, alpha=0.6, ax=axes[1])
axes[1].set_title('Verklig malignitet (facit)')

plt.tight_layout()
plt.show()

Jämför de två crosstab-tabellerna och de två scatterplotten sida vid sida. Om klustren till stor del matchar den verkliga malignitetsetiketten, eller åtminstone samlar liknande diagnoser tillsammans, är det ett tecken på att modellen lärt sig en genuint meningsfull representation, inte bara en godtycklig beslutsgräns.

## 10. Extra: identifiera bilder som inte föreställer hudlesioner
En modell tränad enbart på hudlesioner har inget sätt att säga "jag vet inte", visas den en helt orelaterad bild (en semesterbild, en skärmdump) tvingas den ändå placera bilden någonstans på sin lärda skala. I en app där riktiga användare laddar upp riktiga bilder är det ett verkligt problem, inte ett teoretiskt. Testat och dokumenterat här, inklusive var metoden faktiskt inte räcker till.

In [ ]:
from sklearn.neighbors import NearestNeighbors
import joblib

k_neighbors = 10
nn_ood_model = NearestNeighbors(n_neighbors=k_neighbors + 1)
nn_ood_model.fit(embeddings_scaled)

distances_train, _ = nn_ood_model.kneighbors(embeddings_scaled)
avg_knn_distances = distances_train[:, 1:].mean(axis=1)  # exkluderar avstånd 0 till sig själv

ood_threshold_knn = np.percentile(avg_knn_distances, 99)
print(f"99:e percentilen av medelavstånd till 10 närmaste grannar: {ood_threshold_knn:.3f}")

Idén: mät hur långt en ny bild ligger, i den tränade modellens interna representation, från sina 10 mest lika riktiga träningsbilder. En äkta hudlesion, även en ovanlig sådan, borde ha flera genuint lika bilder någonstans bland de 10015 träningsbilderna. En bild av något helt orelaterat borde inte det. `[:, 1:]` utesluter varje bilds avstånd till sig själv (alltid 0), annars skulle tröskeln bli konstgjort låg.

In [ ]:
(cifar_x, _), (_, _) = tf.keras.datasets.cifar10.load_data()

sample_idx = np.random.RandomState(42).choice(len(cifar_x), 500, replace=False)
cifar_sample = cifar_x[sample_idx]

cifar_resized = np.array([
    np.array(Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
    for img in cifar_sample
])

cifar_embeddings = feature_extractor.predict(cifar_resized, batch_size=32, verbose=1)
cifar_embeddings_scaled = scaler.transform(cifar_embeddings)

cifar_distances, _ = nn_ood_model.kneighbors(cifar_embeddings_scaled)
cifar_avg_distances = cifar_distances.mean(axis=1)

print("Hudbilder, medelavstånd:")
print(f"  Median: {np.median(avg_knn_distances):.2f}, 99:e percentilen: {np.percentile(avg_knn_distances, 99):.2f}")
print("CIFAR-10 (definitivt inte hud), medelavstånd:")
print(f"  Median: {np.median(cifar_avg_distances):.2f}, 1:a percentilen: {np.percentile(cifar_avg_distances, 1):.2f}")

CIFAR-10 (bilar, fåglar, djur, vardagsobjekt) används som ett gratis, färdigt facit på "definitivt inte hud", inget eget urval behöver samlas in för att testa detta ärligt.

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(avg_knn_distances, bins=40, alpha=0.6, label='Hudbilder (träning)', color='#2a78d6')
plt.hist(cifar_avg_distances, bins=40, alpha=0.6, label='CIFAR-10 (ej hud)', color='#e34948')
plt.xlabel('Medelavstånd till 10 närmaste grannar')
plt.legend()
plt.title('Separerar avståndet hud från icke-hud?')
plt.show()

Hudbilder klustrar tätt kring ett medianavstånd på 8,67. CIFAR-10 sprider sig betydligt bredare, från runt 8,5 och uppåt förbi 40 till 50, utan en tydlig egen topp. Det avgörande: CIFAR-10:s 1:a percentil (8,50) ligger i praktiken exakt vid hudbildernas median (8,67). En meningsfull andel av CIFAR-10 landar alltså i samma avståndsintervall som helt vanliga hudbilder, ingen tröskel på den här axeln separerar grupperna helt rent, eftersom det inte finns någon riktig lucka att dra en gräns i.

Detta bekräftar, med data snarare än enstaka exempel, ett verkligt fynd: avstånd i en fryst, ImageNet-förtränad representation mäter generell visuell likhet (textur, kontrast, färgfördelning), inte specifikt "är det här hud". En mörk stadionbild och en strukturerad tapet (två exempel som testades manuellt i den byggda applikationen, en fotbollsarena i skymning och en abstrakt tapetbild) råkar dela tillräckligt av dessa lågnivåegenskaper med dermatoskopiska bilder för att glida igenom, medan ett foto av en person (varierat innehåll, kläder, bakgrund, hud och hår tillsammans i samma bild) korrekt flaggades som orelaterat.

In [ ]:
joblib.dump(scaler, '/content/embedding_scaler.pkl')
joblib.dump(nn_ood_model, '/content/knn_ood_model.pkl')
joblib.dump(ood_threshold_knn, '/content/ood_threshold_knn.pkl')

from google.colab import files
files.download('/content/embedding_scaler.pkl')
files.download('/content/knn_ood_model.pkl')
files.download('/content/ood_threshold_knn.pkl')

Beslut: kontrollen behålls som ett bästa-möjliga-ansträngning-skyddsnät i applikationen, inte som en garanti, och begränsningen dokumenteras öppet snarare än att presenteras som löst. Den fångar tydligt orelaterat innehåll i de flesta fall och släpper aldrig igenom en riktig hudbild av misstag (ingen falsk avvisning har observerats), men ett smalt band av visuellt lika, innehållsmässigt orelaterade bilder kan glida igenom. En mer robust lösning, en binär klassificerare tränad explicit på hud mot icke-hud som negativa exempel, är en tydlig, avgränsad utökning för framtiden, utanför vad som byggs här nu.

## 11. Strukturerad data: Random Forest på metadata

In [ ]:
cat_cols = ['sex', 'localization']
X_structured = pd.get_dummies(df[['age'] + cat_cols], columns=cat_cols, drop_first=True)
y_structured = df['malignant']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_structured, y_structured, test_size=0.2, random_state=42, stratify=y_structured
)
print(X_train_s.shape, X_test_s.shape)

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
rf.fit(X_train_s, y_train_s)

y_pred_rf = rf.predict(X_test_s)
y_pred_proba_rf = rf.predict_proba(X_test_s)[:, 1]

print(confusion_matrix(y_test_s, y_pred_rf))
print(classification_report(y_test_s, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test_s, y_pred_proba_rf))

precision_rf, recall_rf, _ = precision_recall_curve(y_test_s, y_pred_proba_rf)
pr_auc_rf = auc(recall_rf, precision_rf)
print("PR-AUC:", pr_auc_rf)

`class_weight='balanced'` är motiverat annorlunda här än för CNN:et. För en Random Forest justerar viktningen direkt hur splittar väljs vid trädbygget, ett mer direkt och mindre förvrängningsbenäget mekanism än hos ett djupt nätverk, och är standardpraxis för tabelldata specifikt.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train_s.columns).sort_values(ascending=False)
print(importances.head(10))

In [ ]:
print(df.groupby('localization')['malignant'].mean().sort_values(ascending=False))

`age` står för majoriteten av modellens vikt (cirka 64%), konsistent med det 14-åriga gapet från EDA:n. `localization_trunk` rankas högt (cirka 10%) trots att bål har en av de lägsta malignitetsandelarna (4,2%), inte en motsägelse, utan ett genuint negativt samband: bål-lokalisation drar prediktionen mot benign, en riktig, förklarbar signal snarare än brus.

## 12. Kombinerad modell: bild och metadata tillsammans
Både bilden och de strukturerade patientfakta (ålder, kön, lokalisation) bär på riktig signal var för sig. Frågan här är om en modell som ser båda samtidigt presterar bättre än endera ensam.

In [ ]:
cat_cols = ['sex', 'localization']
metadata_encoded = pd.get_dummies(df[['age'] + cat_cols], columns=cat_cols, drop_first=True)

meta_train = metadata_encoded.loc[train_df.index].values
meta_test = metadata_encoded.loc[test_df.index].values

meta_scaler = StandardScaler()
meta_train_scaled = meta_scaler.fit_transform(meta_train)
meta_test_scaled = meta_scaler.transform(meta_test)

embeddings_train = embeddings[:len(train_df)]
embeddings_test = embeddings[len(train_df):]

print(embeddings_train.shape, meta_train_scaled.shape)
print(embeddings_test.shape, meta_test_scaled.shape)

Två val värda att förstå. Modellen återanvänder embeddings, den redan tränade bildrepresentationen från avsnitt 9, istället för att bygga en ny bildgren från grunden, tränar på sekunder istället för minuter eftersom inga bilder körs genom EfficientNet igen. Metadatan hämtas explicit via .loc[train_df.index], samma rader som embeddings faktiskt kommer från, snarare än att lita på att en separat train_test_split-uppdelning råkar landa på exakt samma rader. StandardScaler på metadatan är nödvändigt eftersom age (0 till 85) och de binära dummy-kolumnerna annars skulle vara på helt olika skalor.

In [ ]:
image_embed_input = layers.Input(shape=(128,), name='image_embedding')
meta_input = layers.Input(shape=(meta_train_scaled.shape[1],), name='metadata')

x_meta = layers.Dense(32, activation='relu')(meta_input)
x_meta = layers.Dropout(0.2)(x_meta)

combined = layers.Concatenate()([image_embed_input, x_meta])
combined = layers.Dense(64, activation='relu')(combined)
combined = layers.Dropout(0.3)(combined)
output = layers.Dense(1, activation='sigmoid')(combined)

combined_model = models.Model(inputs=[image_embed_input, meta_input], outputs=output)
combined_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
combined_model.summary()

In [ ]:
early_stop_combined = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history_combined = combined_model.fit(
    [embeddings_train, meta_train_scaled], y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop_combined],
    verbose=1
)

In [ ]:
y_pred_proba_combined = combined_model.predict([embeddings_test, meta_test_scaled]).flatten()

precision_c, recall_c, _ = precision_recall_curve(y_test, y_pred_proba_combined)
pr_auc_c = auc(recall_c, precision_c)

print(confusion_matrix(y_test, (y_pred_proba_combined >= 0.5).astype(int)))
print(classification_report(y_test, (y_pred_proba_combined >= 0.5).astype(int)))
print(f"PR-AUC (bild + metadata): {pr_auc_c:.3f}")
print(f"PR-AUC (bara bild): {pr_auc:.3f}")
print(f"PR-AUC (bara metadata): {pr_auc_rf:.3f}")

### Är skillnaden verklig, eller bara brus?
Samma CNN-arkitektur gav 0,660 respektive 0,625 i PR-AUC i två separata körningar tidigare, enbart på grund av slumpmässig viktinitiering, ingen annan skillnad. En enda jämförelse här räcker därför inte. Eftersom den kombinerade modellen tränar på redan extraherade embeddings, sekunder per epok snarare än minuter, är det billigt att köra om flera gånger och faktiskt mäta variationen istället för att anta.

In [ ]:
combined_prauc_scores = []

for run in range(5):
    tf.random.set_seed(run)

    image_embed_input_r = layers.Input(shape=(128,), name='image_embedding')
    meta_input_r = layers.Input(shape=(meta_train_scaled.shape[1],), name='metadata')

    x_meta_r = layers.Dense(32, activation='relu')(meta_input_r)
    x_meta_r = layers.Dropout(0.2)(x_meta_r)

    combined_r = layers.Concatenate()([image_embed_input_r, x_meta_r])
    combined_r = layers.Dense(64, activation='relu')(combined_r)
    combined_r = layers.Dropout(0.3)(combined_r)
    output_r = layers.Dense(1, activation='sigmoid')(combined_r)

    model_run = models.Model(inputs=[image_embed_input_r, meta_input_r], outputs=output_r)
    model_run.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    early_stop_run = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
    model_run.fit(
        [embeddings_train, meta_train_scaled], y_train,
        validation_split=0.2, epochs=30, batch_size=32,
        callbacks=[early_stop_run], verbose=0
    )

    y_pred_proba_run = model_run.predict([embeddings_test, meta_test_scaled], verbose=0).flatten()
    precision_run, recall_run, _ = precision_recall_curve(y_test, y_pred_proba_run)
    pr_auc_run = auc(recall_run, precision_run)
    combined_prauc_scores.append(pr_auc_run)
    print(f"Run {run}: PR-AUC = {pr_auc_run:.3f}")

print()
print(f"Medel: {np.mean(combined_prauc_scores):.3f}, std: {np.std(combined_prauc_scores):.3f}")
print(f"Spann: {min(combined_prauc_scores):.3f} till {max(combined_prauc_scores):.3f}")

Fem körningar landar tätt (std 0,003, spann 0,601 till 0,611), betydligt tightare än bildmodellens egen körning-till-körning-variation (0,035). Men hela det spannet ligger tydligt under bildmodellens 0,625, en skillnad på 5 till 8 standardavvikelser, ett äkta, konsekvent resultat, inte slump. Kombinationen presterar alltså sämre än bilden ensam, trots att metadatan bär riktig signal isolerat (Random Forest, 0,436 PR-AUC).

Trolig förklaring: embeddings är redan tränade på riktiga bilder av lesioner, och lesionens utseende korrelerar redan indirekt med sådant som kroppsdel. Metadatagrenen tillför då delvis redundant, inte ny, information, samtidigt som den lägger till fler parametrar för optimeraren att hantera. Intressant sidofynd: den kombinerade modellen konvergerar betydligt mer stabilt (std 0,003 mot 0,035), även om den inte konvergerar till ett bättre resultat.

Beslut: cnn_model (bild ensam) förblir modellen som går vidare till serveringen. Fem körningar är tillräckligt robust bevis för att stänga det här spåret ärligt, istället för att jaga ett sjätte försök i hopp om ett annat svar.

## 13. Jämförelse och slutsats

In [ ]:
comparison = pd.DataFrame({
    'Modell': ['CNN (bild)', 'Random Forest (metadata)', 'Kombinerad (bild + metadata)'],
    'PR-AUC': [pr_auc, pr_auc_rf, np.mean(combined_prauc_scores)],
    'ROC-AUC': [roc_auc_score(y_test, y_pred_proba), roc_auc_score(y_test_s, y_pred_proba_rf), roc_auc_score(y_test, y_pred_proba_combined)]
})
print(comparison)

Bilden ensam vinner, konsekvent över upprepade körningar. Strukturerad data ensam (0,436 PR-AUC) fångar redan meningsfull risk, konsistent med EDA:ns fynd om ålder och solexponering, men att kombinera de två på det här sättet tillförde inte mer, embeddings verkar redan implicit fånga mycket av det metadatan skulle bidra med.

Slutgiltig modell: cnn_model, fas 1, fryst EfficientNetB0-bas, tröskel satt till 90% recall. Två alternativa träningsstrategier och en kombinerad arkitektur testades och avfärdades med upprepade mätningar, inte enstaka körningar eller antagande. Ett resultat som inte förbättrades är fortfarande ett resultat värt att redovisa ärligt.

## 14. Spara modellen (första versionen)

In [ ]:
cnn_model.save('/content/skin_cnn_final.keras')

from google.colab import files
files.download('/content/skin_cnn_final.keras')

## 15. Utökning: verkliga fotografier (PAD-UFES-20)
Ett informellt test mot fem bekräftade melanombilder, fotograferade på vanligt sätt snarare än med dermatoskop, visade att modellen ovan missade tre av fem, alla under 10% sannolikhet. HAM10000 innehåller uteslutande dermatoskopiska bilder, modellen har aldrig sett en vanlig hudbild under träning. Testat och åtgärdat här, inte bara noterat.

In [ ]:
import kagglehub
pad_path = kagglehub.dataset_download("orvile/pad-ufes-20")
print(pad_path)
print(os.listdir(pad_path))

Betydligt enklare struktur än HAM10000, en `images`-mapp och en `metadata.csv`, ingen uppdelning i flera CSV-varianter att sålla bort.

In [ ]:
images_path = os.path.join(pad_path, 'images')
print(os.listdir(images_path))

pad_df = pd.read_csv(os.path.join(pad_path, 'metadata.csv'))
print(pad_df.shape)
pad_df[['diagnostic', 'biopsed', 'img_id']].head()

### Verifiera datakvaliteten innan den litas på
Dokumentationen påstår att 100% av cancerfallen är biopsibekräftade. Kontrolleras direkt mot datan istället för att tas för givet.

In [ ]:
print(pad_df.groupby('diagnostic')['biopsed'].mean())

BCC, MEL och SCC ligger alla på exakt 100% biopsibekräftade, ACK, NEV och SEK betydligt lägre (24 till 6%). Påståendet stämmer, verifierat, inte bara citerat.

### En definitionskonflikt att lösa medvetet
PAD-UFES-20:s egen dokumentation klassar aktinisk keratos (ACK) som en av de icke-cancerösa kategorierna. HAM10000-pipelinen ovan klassar samma tillstånd (`akiec`) som malignt, eftersom det är ett cancerförstadium. Motsägande etiketter för samma diagnos hade varit en dold datakvalitetsbrist i den kombinerade datan. Löst genom att hålla fast vid den redan etablerade, försiktigare definitionen: ACK räknas som malignt, konsekvent med `akiec` och med projektets genomgående princip att en missad cancer väger tyngre än ett onödigt läkarbesök.

In [ ]:
malignant_pad = ['BCC', 'MEL', 'SCC']
pad_df['malignant'] = pad_df['diagnostic'].isin(malignant_pad).astype(int)
print(pad_df['malignant'].value_counts())
print(pad_df['malignant'].value_counts(normalize=True).round(3) * 100)

47,4% malignt i PAD-UFES-20, mot 19,5% i HAM10000. Väntat, det här är ett kliniskt insamlat dataset med högre andel remitterade, misstänkta fall, inte ett stickprov av alla hudförändringar i befolkningen.

### En bugg i bildsökvägarna, hittad genom att kontrollera, inte anta

In [ ]:
pad_image_paths = {}
for part in ['imgs_part_1', 'imgs_part_2', 'imgs_part_3']:
    part_dir = os.path.join(images_path, part)
    for fname in os.listdir(part_dir):
        pad_image_paths[fname] = os.path.join(part_dir, fname)

pad_df['image_path'] = pad_df['img_id'].map(pad_image_paths)
print("Saknade bildsökvägar:", pad_df['image_path'].isnull().sum())

Samtliga 2298 rader saknade bildsökväg vid första försöket, inte några enstaka, ett tecken på ett strukturellt fel snarare än enskilda saknade filer. `os.listdir()` på en av delmapparna visade att den bara innehöll ännu en mapp med samma namn, en nästlad struktur, troligen ett resultat av hur arkivet packades. Kontrollerad och åtgärdad nedan.

In [ ]:
pad_image_paths = {}
for part in ['imgs_part_1', 'imgs_part_2', 'imgs_part_3']:
    part_dir = os.path.join(images_path, part, part)  # nästlad mapp, samma namn två gånger
    for fname in os.listdir(part_dir):
        pad_image_paths[fname] = os.path.join(part_dir, fname)

pad_df['image_path'] = pad_df['img_id'].map(pad_image_paths)
print("Saknade bildsökvägar:", pad_df['image_path'].isnull().sum())
print(pad_df.shape)

### Kombinera datasetet med HAM10000

In [ ]:
ham_subset = df[['image_path', 'malignant']].copy()
ham_subset['source'] = 'HAM10000'

pad_subset = pad_df[['image_path', 'malignant']].copy()
pad_subset['source'] = 'PAD-UFES-20'

combined_df = pd.concat([ham_subset, pad_subset], ignore_index=True)
print(combined_df.shape)
print(combined_df['source'].value_counts())
print(combined_df.groupby('source')['malignant'].mean().round(3))

`source`-kolumnen är inte bara bokföring, den är förutsättningen för att senare kunna utvärdera de två bildstilarna separat, den enda riktiga proven på om modellen faktiskt blivit bättre på verkliga foton, inte bara ett antagande.

In [ ]:
combined_train_df, combined_test_df = train_test_split(
    combined_df, test_size=0.2, random_state=42,
    stratify=combined_df[['malignant', 'source']].astype(str).agg('_'.join, axis=1)
)
print(combined_train_df.shape, combined_test_df.shape)
print(combined_train_df.groupby('source')['malignant'].mean().round(3))
print(combined_test_df.groupby('source')['malignant'].mean().round(3))

Stratifiering på både malignitet och källa samtidigt, inte bara malignitet. Att bara stratifiera på malignitet hade kunnat lämna käll-fördelningen skev av slump, till exempel ett testset som råkar bli 90% HAM10000, vilket hade försvagat just den käll-specifika utvärderingen nedan.

In [ ]:
X_train_combined = np.array([load_and_preprocess(p) for p in combined_train_df['image_path']])
y_train_combined = combined_train_df['malignant'].values

X_test_combined = np.array([load_and_preprocess(p) for p in combined_test_df['image_path']])
y_test_combined = combined_test_df['malignant'].values

print(X_train_combined.shape, X_test_combined.shape)

### Träna en ny modell på den kombinerade datan

In [ ]:
combined_model_path = os.path.join(DRIVE_DIR, 'cnn_model_combined.keras')

if os.path.exists(combined_model_path):
    cnn_model_combined = tf.keras.models.load_model(combined_model_path)
    print("Laddade befintlig kombinerad modell från Drive, ingen träning behövdes.")
else:
    inputs_c = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x_c = layers.Rescaling(255.0)(inputs_c)
    x_c = base_model(x_c)
    x_c = layers.GlobalAveragePooling2D()(x_c)
    x_c = layers.Dense(128, activation='relu', name='embedding_layer')(x_c)
    x_c = layers.Dropout(0.3)(x_c)
    outputs_c = layers.Dense(1, activation='sigmoid')(x_c)

    cnn_model_combined = models.Model(inputs=inputs_c, outputs=outputs_c)
    cnn_model_combined.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    early_stop_c = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
    cnn_model_combined.fit(
        X_train_combined, y_train_combined,
        validation_split=0.2, epochs=20, batch_size=32,
        callbacks=[early_stop_c], verbose=1
    )
    cnn_model_combined.save(combined_model_path)
    print("Tränade och sparade kombinerad modell till Drive.")

Ny, dedikerad variabel, `cnn_model_combined`, inte en återanvändning av `cnn_model`. Den ursprungliga modellen förblir orörd, både i minnet och på Drive, exakt den disciplin som hade förhindrat den allra första bugghändelsen i det här projektet.

### Utvärdera per bildkälla, inte bara totalt
En samlad siffra kan dölja att modellen blivit bättre på ena bildstilen och sämre på den andra. Bryts ner explicit.

In [ ]:
y_pred_proba_combined_model = cnn_model_combined.predict(X_test_combined).flatten()

precision_comb, recall_comb, _ = precision_recall_curve(y_test_combined, y_pred_proba_combined_model)
pr_auc_comb = auc(recall_comb, precision_comb)
print(f"Hela testsetet, PR-AUC: {pr_auc_comb:.3f}, ROC-AUC: {roc_auc_score(y_test_combined, y_pred_proba_combined_model):.3f}")

source_array = combined_test_df['source'].values
for src in ['HAM10000', 'PAD-UFES-20']:
    mask = source_array == src
    precision_src, recall_src, _ = precision_recall_curve(y_test_combined[mask], y_pred_proba_combined_model[mask])
    pr_auc_src = auc(recall_src, precision_src)
    print(f"{src} (n={mask.sum()}): PR-AUC {pr_auc_src:.3f}, ROC-AUC {roc_auc_score(y_test_combined[mask], y_pred_proba_combined_model[mask]):.3f}")

HAM10000-delen (0,634 PR-AUC) ligger i praktiken kvar där den ursprungliga modellen låg (0,655 i sin senaste körning), skillnaden ryms inom den run-till-run-variation som redan uppmätts (upp till 0,035 mellan identiska körningar). Den ursprungliga uppgiften försämrades alltså inte nämnvärt. PAD-UFES-20-delen (0,797 PR-AUC) är stark i sin egen rätt, högre än modellens resultat på sin egen ursprungliga testdata. Modellen tolererar inte bara vanliga foton nu, den presterar jämförbart eller bättre på dem.

### Det avgörande testet: samma sex verkliga bilder som avslöjade problemet

In [ ]:
for path, label in [
    ('/content/melanoma.png', 'melanoma.png'),
    ('/content/melanoma2.webp', 'melanoma2.webp'),
    ('/content/melanoma3.webp', 'melanoma3.webp'),
    ('/content/melanom4.webp', 'melanoma4.webp'),
    ('/content/melanoma5.png', 'melanoma5.png'),
    ('/content/notmelanoma.png', 'notmelanoma.png'),
]:
    arr = load_and_preprocess(path)
    prob = cnn_model_combined.predict(np.expand_dims(arr, 0), verbose=0)[0][0]
    print(f"{label}: {prob:.4f}")

Jämförelse mot den ursprungliga modellens resultat:

| Bild | Ursprunglig modell | Kombinerad modell |
|---|---|---|
| melanoma.png | 7,0% | 19,0% |
| melanoma2.webp | 8,2% | 13,2% |
| melanoma3.webp | 2,2% | 19,1% |
| melanoma4.webp | 34,5% | 7,7% |
| melanoma5.png | 87,2% | 85,3% |
| notmelanoma.png | 14,3% | 39,2% |

Fyra av fem verkliga melanom rörde sig i rätt riktning, i vissa fall kraftigt (2,2% till 19,1%). Inget enskilt fall passerar den gamla binära tröskeln på 0,148 själv, men i det fyrnivåsystem som redan byggts skulle melanoma.png, melanoma2 och melanoma3 nu hamna i “Överväg läkarbedömning” istället för “Låg risk”, en verklig eskalering för tre äkta cancerfall som den ursprungliga modellen tyst vinkade igenom. Ett fall (melanoma4) rörde sig åt fel håll, värt att vara ärlig om snarare än att dölja, sex bilder är för få för att förvänta sig entydig förbättring i varje enskilt fall. Den godartade bilden steg också (14,3% till 39,2%), den nya modellen är genomgående något mindre säker, inte bara mer känslig för cancer specifikt, en ärlig nyansering snarare än en ren vinst.

### Slutsats
Den kombinerade modellen bevarar prestandan på HAM10000, presterar starkt på verkliga foton, och flyttar verkliga, tidigare missade melanom i rätt riktning på det exakta test som avslöjade problemet. `cnn_model_combined` blir den nya, slutgiltiga modellen, ersätter `cnn_model` i allt som följer, inklusive applikationens backend.

## 16. Spara den slutgiltiga, kombinerade modellen

In [ ]:
cnn_model_combined.save('/content/skin_cnn_combined_final.keras')

from google.colab import files
files.download('/content/skin_cnn_combined_final.keras')